# 02 Training — YOLOv8n

This notebook trains and resumes a YOLOv8n object detector on the manually annotated 4-class VOC subset. It is designed for Google Colab with Google Drive persistence, so checkpoints and outputs survive session resets.

# YOLOv8n Fine-Tuning

This notebook is designed to be executed in **Google Colab**. It performs fine-tuning of the lightweight CNN-based YOLOv8n model on the PASCAL VOC subset.

## Google Colab Setup

1. **Hardware Accelerator**: Go to *Runtime > Change runtime type* and select **T4 GPU**.

2. **Google Drive Integration**: This notebook requires access to your Google Drive to load the dataset and save checkpoints. When prompted, grant access to mount `/content/drive`.

## Expected Drive Structure

Before running this notebook, ensure your Google Drive contains the following structure:

    MyDrive/
    └── object_detection_yolov8n/
        └── training_bundle.zip

## Configuration Summary

* **Model**: `yolov8n.pt`
* **Image Size**: 640x640
* **Batch Size**: 16 (Can be reduced to 8 if memory limits are exceeded)
* **Epochs**: 60 (with early stopping patience of 20)
* **Optimizer**: Auto (Default learning rate: 0.01)

In [1]:
!nvidia-smi

Tue Mar 10 06:11:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — install

!pip install -q ultralytics pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.9 MB/s eta 0:00:00


In [3]:
# Cell 3 — imports and Drive mount

from pathlib import Path
import os
import shutil
import zipfile
import json
import time

import yaml
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# Cell 4 — config (YOLO-specific)

# ===== USER CONFIG =====
DRIVE_ROOT = Path("/content/drive/MyDrive/object_detection_yolov8n")
BUNDLE_ZIP_PATH = DRIVE_ROOT / "training_bundle.zip"
WORKSPACE_DIR = Path("/content/workspace_yolov8n")
EXTRACT_DIR = WORKSPACE_DIR / "project_bundle"

MODEL_NAME = "yolov8n.pt"
MODEL_KIND = "yolo"
RUN_PROJECT = str(DRIVE_ROOT / "runs")
RUN_NAME = "yolov8n_voc4"

SEED = 42
IMG_SIZE = 640
EPOCHS_TARGET = 60          # first target
BATCH_SIZE = 16            # reduce to 8 if Colab GPU memory complains
PATIENCE = 20
OPTIMIZER = "auto"         # safe default in Ultralytics
LR0 = 0.01                 # YOLO default-scale is reasonable as a first run
DEVICE = 0

# Optional: overwrite extracted bundle if needed
FORCE_REEXTRACT_BUNDLE = False

In [5]:
# Cell 5 — helpers

def bytes_to_gb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)


def print_disk_usage(path: Path) -> None:
    usage = shutil.disk_usage(path)
    print(f"Disk usage for {path}:")
    print(f"  total: {bytes_to_gb(usage.total):.2f} GB")
    print(f"  used : {bytes_to_gb(usage.used):.2f} GB")
    print(f"  free : {bytes_to_gb(usage.free):.2f} GB")


def extract_bundle_if_needed(bundle_zip_path: Path, extract_dir: Path, force: bool = False) -> None:
    if not bundle_zip_path.exists():
        raise FileNotFoundError(f"Bundle zip not found: {bundle_zip_path}")

    expected_yaml = extract_dir / "data" / "data.yaml"

    if expected_yaml.exists() and not force:
        print(f"Bundle already extracted: {extract_dir}")
        return

    if force and extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(bundle_zip_path, "r") as zf:
        zf.extractall(extract_dir)

    if not expected_yaml.exists():
        raise FileNotFoundError(f"Extraction finished but missing data.yaml at: {expected_yaml}")

    print(f"Bundle extracted to: {extract_dir}")


def read_data_yaml(data_yaml_path: Path) -> dict:
    with open(data_yaml_path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def find_existing_run_dir(run_project: str, run_name: str) -> Path:
    run_dir = Path(run_project) / run_name
    return run_dir


def get_checkpoint_paths(run_dir: Path) -> dict:
    weights_dir = run_dir / "weights"
    return {
        "run_dir": run_dir,
        "weights_dir": weights_dir,
        "last": weights_dir / "last.pt",
        "best": weights_dir / "best.pt",
    }


def checkpoint_exists(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size > 0

In [6]:
# Cell 6 — workspace setup and space check

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

print_disk_usage(Path("/content"))
print_disk_usage(DRIVE_ROOT)

if not BUNDLE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Upload training_bundle.zip to Google Drive first.\nMissing: {BUNDLE_ZIP_PATH}"
    )

print(f"Bundle zip size: {bytes_to_gb(BUNDLE_ZIP_PATH.stat().st_size):.2f} GB")

Disk usage for /content:
  total: 112.64 GB
  used : 43.51 GB
  free : 69.11 GB
Disk usage for /content/drive/MyDrive/object_detection_yolov8n:
  total: 15.00 GB
  used : 0.04 GB
  free : 14.96 GB
Bundle zip size: 0.03 GB


In [7]:
# Cell 7 — extract bundle

extract_bundle_if_needed(
    bundle_zip_path=BUNDLE_ZIP_PATH,
    extract_dir=EXTRACT_DIR,
    force=FORCE_REEXTRACT_BUNDLE,
)

DATA_YAML_PATH = EXTRACT_DIR / "data" / "data.yaml"
data_yaml = read_data_yaml(DATA_YAML_PATH)

print("Loaded data.yaml:")
print(json.dumps(data_yaml, indent=2))

Bundle extracted to: /content/workspace_yolov8n/project_bundle
Loaded data.yaml:
{
  "path": "F:\\Projects\\Object_Detection_Computer_Vision\\data",
  "train": "images/train",
  "val": "images/val",
  "test": "images/test",
  "names": {
    "0": "car",
    "1": "bus",
    "2": "bicycle",
    "3": "motorbike"
  }
}


In [8]:
# Cell 8 — patch data.yaml for Colab runtime

patched_data_yaml = {
    "path": str((EXTRACT_DIR / "data").resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": data_yaml["names"],
}

PATCHED_DATA_YAML_PATH = EXTRACT_DIR / "data" / "data_colab.yaml"

with open(PATCHED_DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(patched_data_yaml, f, sort_keys=False, allow_unicode=True)

print(f"Patched Colab data.yaml written to: {PATCHED_DATA_YAML_PATH}")
print(json.dumps(patched_data_yaml, indent=2))

Patched Colab data.yaml written to: /content/workspace_yolov8n/project_bundle/data/data_colab.yaml
{
  "path": "/content/workspace_yolov8n/project_bundle/data",
  "train": "images/train",
  "val": "images/val",
  "test": "images/test",
  "names": {
    "0": "car",
    "1": "bus",
    "2": "bicycle",
    "3": "motorbike"
  }
}


In [9]:
# Cell 9 — checkpoint discovery

run_dir = find_existing_run_dir(RUN_PROJECT, RUN_NAME)
ckpt = get_checkpoint_paths(run_dir)

print("Run directory:", ckpt["run_dir"])
print("Last checkpoint exists:", checkpoint_exists(ckpt["last"]))
print("Best checkpoint exists:", checkpoint_exists(ckpt["best"]))

Run directory: /content/drive/MyDrive/object_detection_yolov8n/runs/yolov8n_voc4
Last checkpoint exists: False
Best checkpoint exists: False


In [10]:
# Cell 10 — pre-training evaluation helper

def validate_model(model_path_or_name, data_yaml_path, split="test"):
    model = YOLO(str(model_path_or_name))
    metrics = model.val(
        data=str(data_yaml_path),
        split=split,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        plots=False,
    )
    return metrics


def metrics_to_dict(metrics, label: str) -> dict:
    return {
        "label": label,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "mp": float(metrics.box.mp),
        "mr": float(metrics.box.mr),
    }

In [11]:
# Cell 11 — evaluate existing checkpoint before training if present

pretrain_metrics = None

if checkpoint_exists(ckpt["best"]):
    print("Evaluating existing best checkpoint before training...")
    metrics_before = validate_model(
        model_path_or_name=ckpt["best"],
        data_yaml_path=PATCHED_DATA_YAML_PATH,
        split="test",
    )
    pretrain_metrics = metrics_to_dict(metrics_before, label="before_training")
    print(pretrain_metrics)
else:
    print("No existing best checkpoint found. Fresh training run will start from pretrained weights.")

No existing best checkpoint found. Fresh training run will start from pretrained weights.


In [12]:
# Cell 12 — training cell (YOLO)

if checkpoint_exists(ckpt["last"]):
    print(f"Resuming training from: {ckpt['last']}")
    model = YOLO(str(ckpt["last"]))
    train_results = model.train(
        data=str(PATCHED_DATA_YAML_PATH),
        epochs=EPOCHS_TARGET,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        device=DEVICE,
        project=RUN_PROJECT,
        name=RUN_NAME,
        optimizer=OPTIMIZER,
        lr0=LR0,
        patience=PATIENCE,
        resume=True,
        save=True,
        save_period=-1,
        plots=True,
    )
else:
    print(f"Starting fresh training from pretrained weights: {MODEL_NAME}")
    model = YOLO(MODEL_NAME)
    train_results = model.train(
        data=str(PATCHED_DATA_YAML_PATH),
        epochs=EPOCHS_TARGET,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        seed=SEED,
        device=DEVICE,
        project=RUN_PROJECT,
        name=RUN_NAME,
        optimizer=OPTIMIZER,
        lr0=LR0,
        patience=PATIENCE,
        save=True,
        save_period=-1,
        plots=True,
    )

print("Training complete.")

Starting fresh training from pretrained weights: yolov8n.pt
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/workspace_yolov8n/project_bundle/data/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_voc4, nbs=64, nm